In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

lms = spark.read.table("edtech_project.edtech_bronze.lms_events_bronze")

# Ensure correct types
lms = lms.withColumn("event_ts", F.col("event_ts").cast("timestamp"))

# Basic cleaning
lms = lms.filter(
    F.col("student_id").isNotNull() &
    F.col("course_id").isNotNull() &
    F.col("event_ts").isNotNull()
)

In [0]:
from pyspark.sql.functions import col, coalesce, lit

df = lms.withColumn(
    "duration_sec",
    coalesce(col("duration_seconds").cast("int"), lit(0))
)

In [0]:
df.show()

In [0]:
from pyspark.sql.functions import expr

df = df.withColumn(
    "event_end_ts",
    expr("event_ts + interval 1 second * duration_sec")
)

df.show()

In [0]:
from pyspark.sql.window import Window

window_spec = Window.partitionBy("student_id").orderBy("event_ts")

In [0]:
from pyspark.sql.functions import lag

df = df.withColumn(
    "prev_end_ts",
    lag("event_end_ts").over(window_spec)
)

df.show()

In [0]:
from pyspark.sql.functions import unix_timestamp

df = df.withColumn(
    "gap_minutes",
    (unix_timestamp("event_ts") - unix_timestamp("prev_end_ts")) / 60
)

df.show()

In [0]:
 from pyspark.sql.functions import when

df = df.withColumn(
    "is_new_session",
    when(col("gap_minutes") > 30, 1).otherwise(0)
)

df.show()
# SESSION_GAP = 240

# df= df.withColumn(
#     "is_new_session",
#     F.when(
#          (col("gap_minutes") > SESSION_GAP),
#         1
#     ).otherwise(0)
# )

In [0]:
df = df.fillna({"is_new_session": 1})

df.show()

In [0]:
from pyspark.sql.functions import sum as spark_sum

df = df.withColumn(
    "session_id",
    spark_sum("is_new_session").over(window_spec)
)

In [0]:
from pyspark.sql.functions import min, max, count, countDistinct

session_df = df.groupBy("student_id", "session_id", "course_id").agg(
    min("event_ts").alias("session_start"),
    max("event_end_ts").alias("session_end"),
    ((unix_timestamp(max("event_end_ts")) - unix_timestamp(min("event_ts"))) / 60)
        .alias("session_duration_mins"),
    count("*").alias("events_in_session"),
    countDistinct("module_id").alias("modules_touched")
)

In [0]:
from pyspark.sql.functions import desc, row_number
from pyspark.sql.window import Window

activity_df = df.groupBy("student_id", "session_id", "event_type").count()

activity_window = Window.partitionBy("student_id", "session_id") \
    .orderBy(desc("count"))

primary_activity_df = activity_df.withColumn(
    "rank",
    row_number().over(activity_window)
).filter("rank = 1").select(
    "student_id",
    "session_id",
    col("event_type").alias("primary_activity")
)

In [0]:
final_df = session_df.join(
    primary_activity_df,
    ["student_id", "session_id"],
    "left"
)

In [0]:
from pyspark.sql.functions import when, col

final_df = final_df.withColumn(
    "is_suspicious_session",
    when(col("session_duration_mins") > 480, 1).otherwise(0)
)

In [0]:
display(final_df)

In [0]:
final_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("edtech_project.edtech_silver.learning_sessions")

In [0]:
final_df.count()

In [0]:
final_df.select(
    F.min("session_duration_mins"),
    F.avg("session_duration_mins"),
    F.max("session_duration_mins")
).show()

In [0]:
final_df.filter(F.col("session_duration_mins") > 240).count()

In [0]:
final_df.select(
    F.min("events_in_session"),
    F.avg("events_in_session"),
    F.max("events_in_session")
).show()

In [0]:
final_df.groupBy("events_in_session") \
    .count().orderBy("events_in_session").show()

In [0]:
final_df.select("student_id","course_id","session_start","session_duration_mins") \
   .orderBy("student_id","session_start") \
   .show(50, False)

In [0]:
final_df.filter(F.col("session_duration_mins") <= 30).count()

In [0]:
final_df.select(
    "student_id","course_id","session_start","session_duration_mins"
).orderBy("student_id","session_start").show(50, False)